# F1 INSIGHT — 04 · Dynamic Insights

The insight engine computes findings straight from the warehouse — every value
is produced by SQL or numpy at request time (5-minute cache), nothing is
hard-coded. This notebook runs the same code as `GET /api/insights`.

## Setup

In [ ]:
# Convenience setup: make the backend package importable (run from repo root).
import os, sys
ROOT = os.getcwd()
for candidate in (os.path.join(ROOT, "backend"), os.path.join(ROOT, "..", "backend")):
    if os.path.isdir(candidate):
        sys.path.insert(0, candidate)
        break
import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("pandas", pd.__version__)


from sqlalchemy import text
from app.db import SessionLocal


def q(sql, **params):
    "Run SQL against the warehouse and return rows as dicts."
    with SessionLocal() as db:
        rows = db.execute(text(sql), params).mappings().all()
    return [dict(r) for r in rows]


def counts():
    tables = ["dim_date", "dim_driver", "dim_constructor", "dim_circuit", "dim_race",
              "fact_race_result", "fact_qualifying", "fact_lap", "fact_pit_stop",
              "fact_driver_standing", "fact_constructor_standing"]
    out = []
    total = 0
    for t in tables:
        n = q(f"SELECT COUNT(*) AS n FROM {t}")[0]["n"]
        total += n
        out.append((t, n))
    out.append(("TOTAL", total))
    return out

## Generate the insights

Each insight carries a category, title, human-readable text and a numeric
value + unit.

In [ ]:
from app.services.insights import generate_insights

with SessionLocal() as db:
    items = generate_insights(db)

print('insights:', len(items))
for it in items:
    print('[' + it['category'] + ']', it['title'])
    print('    ', it['text'])

## Overview by category

Insights fall into Qualifying / Race Craft / Constructors / Reliability /
Pit Stops / Championships / Telemetry / Scoring buckets depending on the data.

In [ ]:
from collections import Counter

counts = Counter(it['category'] for it in items)
(
    pd.DataFrame(counts.items(), columns=['category', 'insights'])
    .sort_values('insights', ascending=False)
)

## Chart the numeric values

Insights with numeric values (correlations, percentages, points, positions)
plotted by category — a quick visual summary of what the data says.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
cats = sorted(counts)
ax.barh(cats, [counts[c] for c in cats], color="#e10600")
ax.set_title("Insights per category")
ax.set_xlabel("count")

ax2 = axes[1]
vals = [(it['title'], float(it['value'] or 0))
        for it in items if it['value'] is not None]
vals.sort(key=lambda x: x[1])
ax2.barh([v[0][:34] for v in vals], [v[1] for v in vals], color="#949497")
ax2.set_title("Insight numeric values")
ax2.set_xlabel("value")

plt.tight_layout()
plt.show()

## Export summary

Write the insight list to CSV for reporting.

In [ ]:
df_ins = pd.DataFrame([
    {k: it[k] for k in ('id', 'category', 'title', 'text', 'unit')}
    | {'value': float(it['value']) if it['value'] is not None else None}
    for it in items
])
df_ins.to_csv('insights_summary.csv', index=False)
df_ins.head()